# Model Evaluation

Kevin Han, Noah Son, Sophia Fu

Loads preprocessed ECoG spectrograms from `cleaned_data/`, fits a Ridge regression model per subject, and evaluates using the **average Pearson correlation across 12 fingers** (4 fingers × 3 subjects; ring finger skipped). Leaderboard predictions are saved to `predictions/`.

In [ ]:
import sys, pathlib, pickle
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge

PROJECT_ROOT   = pathlib.Path("..").resolve()
CLEAN_DATA_DIR = PROJECT_ROOT / "cleaned_data"
PRED_DIR       = PROJECT_ROOT / "predictions"
PRED_DIR.mkdir(exist_ok=True)

N_SUBJECTS   = 3
N_FINGERS    = 5
EVAL_FINGERS = [0, 1, 2, 4]  # skip ring finger (index 3)
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
TRAIN_FRAC   = 0.8

## Load Cleaned Data

In [ ]:
data = {}

for subj in range(N_SUBJECTS):
    subj_dir = CLEAN_DATA_DIR / f"subj{subj + 1}"
    specs_train = np.load(subj_dir / "specs_train.npy")  # (channels, freqs, T)
    ff_train    = np.load(subj_dir / "ff_train.npy")     # (5, T)
    specs_lead  = np.load(subj_dir / "specs_lead.npy")   # (channels, freqs, T_lead)

    data[subj] = {"specs_train": specs_train, "ff_train": ff_train, "specs_lead": specs_lead}

    print(f"Subject {subj+1}:  specs_train={specs_train.shape}  ff_train={ff_train.shape}  specs_lead={specs_lead.shape}")

## Feature Preparation

Flatten the `(channels, freqs, T)` spectrograms to `(T, channels × freqs)`, then build a **lagged feature matrix** that concatenates the current time step with the `N_LAGS - 1` preceding steps. This lets the linear model exploit the recent history of neural activity — analogous to the R matrix from the optimal linear decoder homework.

In [ ]:
def flatten_specs(specs: np.ndarray) -> np.ndarray:
    """(channels, freqs, T) -> (T, channels * freqs)"""
    channels, freqs, T = specs.shape
    return specs.reshape(channels * freqs, T).T


def create_lagged_features(X: np.ndarray, n_lags: int = 3) -> np.ndarray:
    """
    For each time step t, concatenate features from [t-n_lags+1, ..., t].
    The first n_lags-1 rows are padded by repeating X[0].

    Input:  (T, n_features)
    Output: (T, n_lags * n_features)
    """
    T, n_feats = X.shape
    padding  = np.repeat(X[0:1], n_lags - 1, axis=0)   # (n_lags-1, n_feats)
    X_padded = np.vstack([padding, X])                  # (T + n_lags - 1, n_feats)

    lagged = np.lib.stride_tricks.sliding_window_view(
        X_padded, window_shape=(n_lags, n_feats)
    )[:, 0, :, :].reshape(T, n_lags * n_feats)
    return lagged


def train_val_split(X, y, train_frac=TRAIN_FRAC):
    """Chronological split to preserve temporal structure."""
    cut = int(X.shape[0] * train_frac)
    return X[:cut], X[cut:], y[:cut], y[cut:]


N_LAGS = 3
print(f"Using N_LAGS={N_LAGS}  ->  each sample covers {N_LAGS * 10} ms of history at 100 Hz")

## Train and Evaluate

Ridge regression is fit on the training portion; performance is measured on the held-out validation portion.

**Metric**: average Pearson *r* across 12 fingers (thumb, index, middle, pinky × 3 subjects).

In [ ]:
results    = []
models     = {}
lead_preds = {}

for subj in range(N_SUBJECTS):
    print(f"\n{'='*50}")
    print(f"  Subject {subj + 1}")
    print(f"{'='*50}")

    # Build lagged feature matrix
    X = create_lagged_features(flatten_specs(data[subj]["specs_train"]), n_lags=N_LAGS)
    y = data[subj]["ff_train"].T  # (T, 5)

    X_train, X_val, y_train, y_val = train_val_split(X, y)
    print(f"  Train: {X_train.shape}  Val: {X_val.shape}")

    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    models[subj] = model

    y_val_pred = model.predict(X_val)

    print("  Validation correlations:")
    for f_idx in range(N_FINGERS):
        corr, _ = pearsonr(y_val[:, f_idx], y_val_pred[:, f_idx])
        skip = "  [SKIP]" if f_idx == 3 else ""
        print(f"    {FINGER_NAMES[f_idx]:8s} (finger {f_idx+1}): r = {corr:+.4f}{skip}")
        results.append({"subject": subj+1, "finger": FINGER_NAMES[f_idx], "f_idx": f_idx, "corr": corr})

    # Leaderboard predictions (apply same lagging)
    X_lead = create_lagged_features(flatten_specs(data[subj]["specs_lead"]), n_lags=N_LAGS)
    lead_preds[subj] = model.predict(X_lead)

df = pd.DataFrame(results)
eval_corrs = df[df["f_idx"].isin(EVAL_FINGERS)]["corr"]
print(f"\n{'='*50}")
print(f"  Mean correlation (12 fingers): {eval_corrs.mean():.4f}")
print(f"{'='*50}")

## Per-Subject Summary Table

In [ ]:
pivot = (
    df[df["f_idx"].isin(EVAL_FINGERS)]
    .pivot(index="subject", columns="finger", values="corr")
    [[FINGER_NAMES[i] for i in EVAL_FINGERS]]
)
pivot["Mean"] = pivot.mean(axis=1)
print(pivot.round(4).to_string())
print(f"
Overall mean: {pivot[chr(77)+chr(101)+chr(97)+chr(110)].mean():.4f}")

## Save Leaderboard Predictions

In [ ]:
for subj, preds in lead_preds.items():
    out_path = PRED_DIR / f"subj{subj+1}_leaderboard_pred.npy"
    np.save(out_path, preds)
    print(f"Subject {subj+1}: saved {preds.shape} -> {out_path.name}")